# 1. Libraries

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown as md
from itertools import combinations

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
import seaborn as sns
import seaborn.objects as so

# 2. Data load

In [ ]:
sheet1 = pd.read_excel("data/online_retail_II.xlsx", sheet_name="Year 2009-2010", engine="openpyxl", dtype={"Customer ID" : "Int64"})
sheet2 = pd.read_excel("data/online_retail_II.xlsx", sheet_name="Year 2010-2011", engine="openpyxl", dtype={"Customer ID" : "Int64"})

check1 = sheet1.shape[0]
check2 = sheet2.shape[0]
sum_separate = check1 + check2
sum_separate

# concat
df_raw = pd.concat([sheet1, sheet2], axis=0)
sum_concat = df_raw.shape[0]

# check rows
print(f"Check of sheets concat:{sum_separate == sum_concat}")

## 2.1. Data clean

In [ ]:
# to avoid loading (just in case)
df = df_raw.copy()

# whitespaces remove
df["Description"] = df["Description"].str.strip()

# revenue column
df["Revenue"] = round((df["Quantity"] * df["Price"]),2)

# date column normalize
df["InvoiceDate"] = df["InvoiceDate"].dt.normalize()

# create year-month column
df["InvoiceDate_YYYYMM"] = df["InvoiceDate"].dt.strftime("%Y-%m")

# 3. EDA

## 3.1. KDE

In [ ]:
# KDE
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns

plt.figure(figsize=(11, len(numerical_columns) * 3))
for idx, feature in enumerate(numerical_columns, 1):
    plt.subplot(len(numerical_columns), 2, idx)
    sns.histplot(df[feature], kde=True)
    if feature in ["Quantity", "Price", "Revenue"]:
        plt.title(f"{feature} | Skewness: {round(df[feature].skew(),2)} \n NOTE: logarithmic scale")
        plt.yscale("log")
    else:
        plt.title(f"{feature} | Skewness: {round(df[feature].skew(),2)}") 
    # plt.ticklabel_format(style='plain', axis='y')
    
    
plt.tight_layout()
plt.show()

## 3.2. Basic stats

In [ ]:
# quick info
df.info()

In [ ]:
# columns null check
df.isnull().sum()

In [ ]:
empty_cus_pct = round((df["Customer ID"].isnull().sum() / df["Customer ID"].shape[0]) * 100, 2)
empty_cus_row = df["Customer ID"].isnull().sum()

In [ ]:
# unique values count
df.nunique()

In [ ]:
# unique values % share
round((df.nunique() / df.count()) * 100,2)

In [ ]:
# basic statistic
df.describe().T

### Quick thoughts

In [ ]:
display(md(f"**{empty_cus_pct}%** rows are missing customer data. It is **{empty_cus_row}** transactions."))
display(md("The data set is highly diversified. Very low percentage of unique values. Any column can be used as category."))
display(md(f"Some outliers can be found in [Quantity] and [Price]."))
display(md(f"[Quantity] minimum value is **[{(df.describe().T.iloc[0,2])}]**, 25-50-75 percentiles are **{list(df.describe().T.iloc[0,3:6])}** and maximum value is **[{(df.describe().T.iloc[0,6])}]**."))
display(md(f"[Price] minimum value is **[{(df.describe().T.iloc[2,2])}]**, 25-50-75 percentiles are **{list(df.describe().T.iloc[2,3:6])}** and maximum value is **[{(df.describe().T.iloc[2,6])}]**."))
display(md(f"And further, this is continued in [Revenue] because it is the result of both of these columns."))

### Quantity outliers

In [ ]:
df.sort_values(by="Quantity", key=lambda x: x.abs(), ascending=False).head(10)

In [ ]:
display(md(f"Even if we noticed outliers, it is not like this they are mistake. It is part of business"))

### Price outliers

In [ ]:
df.sort_values(by="Price", key=lambda x: x.abs(), ascending=False).head(10)

In [ ]:
display(md(f"Even if we noticed outliers, it is not like this they are mistake. It is part of business. They are fees, charges, write-offs"))

In [ ]:
# invoices categories?
df["Invoice"].str[0].value_counts(dropna=False)

In [ ]:
categories_dict = {
    "Sales invoice": lambda s: s.astype(str).str.isnumeric(),
    "Credit note": lambda s: s.astype(str).str[0] == "C",
    "Write-off": lambda s: s.astype(str).str[0] == "A"
}

categories_frames = []
for category, rule in categories_dict.items():
    mask = rule(df["Invoice"])
    frame = df[mask].copy()
    frame["Category"] = category
    categories_frames.append(frame)
    
df = pd.concat(categories_frames)
df["Category"] = df["Category"].astype("category")

# 4. Questions

## 4.1. What is the sales trend over time? Are there seasons/months with higher sales?

In [ ]:
# group invoices count per year-month
time_trend_invoices_count = df.drop_duplicates(subset="Invoice")
time_trend_invoices_count = time_trend_invoices_count.groupby("InvoiceDate_YYYYMM").agg(invoice_count=("Invoice","count"))
# sum revenue per year-month
time_trend_revenue_sum = df.groupby("InvoiceDate_YYYYMM").agg(revenue_sum=("Revenue","sum"))

In [ ]:
sns.set_theme(
    style="white"
    )
f, (ax1, ax2) = plt.subplots(2,1, figsize=(10, 12))

(
    so.Plot(
        data=time_trend_invoices_count,
        x="InvoiceDate_YYYYMM",
        y="invoice_count"
        )       
    .add(
        so.Bar(
            alpha=1, 
            color="#191f29"
            )
        )
    .scale(
        y=so.Continuous().tick(every=500)
        )   
    .label(
        title="Invoices over time | Mean: orange line",
        x="Date",
        y="Invoice count",
        legend=False
        )
    .on(ax1)
    .plot()
)

(
    so.Plot(
        data=time_trend_revenue_sum,
        x="InvoiceDate_YYYYMM",
        y="revenue_sum"
        )       
    .add(
        so.Bar(
            alpha=1, 
            color="#191f29"
            )
        )
    .scale(
        y=so.Continuous().tick(every=100000)
        )   
    .label(
        title="Revenue over time | Mean: orange line",
        x="Date",
        y="Revenue sum",
        legend=False
        )
    .on(ax2)
    .plot()
)

ax1.axhline(y=(time_trend_invoices_count["invoice_count"].mean() ), linewidth=2, color='orange', ls=':')
ax1.yaxis.set_tick_params(which="both", left=True, length=4, rotation=0)
ax1.yaxis.grid(True, linestyle="--", alpha=0.7)
ax1.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax1.yaxis.grid(True, linestyle="--", alpha=0.7)
ax1.set_axisbelow(True)

ax2.axhline(y=(time_trend_revenue_sum["revenue_sum"].mean() ), linewidth=2, color='orange', ls=':')
ax2.yaxis.set_tick_params(which="both", left=True, length=4, rotation=0)
ax2.yaxis.grid(True, linestyle="--", alpha=0.7)
ax2.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
ax2.yaxis.grid(True, linestyle="--", alpha=0.7)
ax2.set_axisbelow(True)

plt.tight_layout()
palette = sns.color_palette()
plt.show()

### Answer

In [ ]:
display(md(f'''
  The "Invoices over time" and "Revenue over time" charts show the individual months in which the values above the reference level for the entire period presented was exceeded.\n
  Above the average, the period from the September to the December/January stands out clearly, ending abruptly.\n
  This can easily be linked to the Christmas and New Year period, when many retail sectors experience increased traffic.\n
  Less visible, but still present, is the increase in the number of invoices in May, which then steadily declines until September, when the aforementioned holiday season begins.\n
  There are also individual months outside this period, but this is not seasonal in nature.
  '''))

## 4.2. Who are the top 10% of customers in terms of purchase value? What percentage of total sales do they generate?

In [ ]:
# sum customer revenue
top_10 = df.groupby("Customer ID").agg(customer_revenue=("Revenue", "sum")).sort_values(by="customer_revenue", ascending=False)
# set 10% revenue limit
mask = top_10["customer_revenue"].quantile(0.9) < top_10["customer_revenue"]
# top 10% customers
top_10 = top_10[mask].reset_index()
# transactions of 10% customers
top_10_customers_transactions = pd.merge(df, top_10["Customer ID"], on="Customer ID", how="inner")

# pct of top 10% customers revenue in all customers revenue
top_10_revenue = top_10["customer_revenue"].sum()
total_revenue = df["Revenue"].sum()
top_10_pct = int(round((top_10_revenue / total_revenue) * 100, 0))

# pct of top 10% customers count in all customers count
top_10_count = top_10["Customer ID"].count()
total_count = df["Customer ID"].drop_duplicates().count()
top_10_count_pct = int(round((top_10_count / total_count) * 100, 0))

In [ ]:
top_10_country_count = top_10_customers_transactions[["Customer ID","Country"]].drop_duplicates().value_counts(subset="Country", ascending=False).iloc[0]
top_10_country_count_pct = int(round((top_10_country_count / top_10_count) * 100, 0))
top_10_country = top_10_customers_transactions[["Customer ID","Country"]].drop_duplicates().value_counts(subset="Country", ascending=False).index[0]

top_10_customers_transactions_products = top_10_customers_transactions["Description"].value_counts(ascending=False).reset_index()

### Answer

In [ ]:
display(md(f"Top 10% of customers: which is **{top_10_count}** customers, generates **{int(top_10_revenue)}** revenue. Which is **{top_10_pct}%** of total revenue."))

display(md(f"They are mostly from **{top_10_country}**. **{top_10_country_count}** of them are from there. It is **{top_10_country_count_pct}%** of all top 10% customers."))

display(md(f"**{list(top_10_customers_transactions_products["Description"].head(5))}** are their top 5 favourite products. They bought them the most."))

## 4.3. Market basket analysis: which products are most often purchased together? (simple approach: transactions with multiple items, check for repetitions)

In [ ]:
# transactions with multiple items
transactions_grouped = df
transactions_grouped["item_count"] = df.groupby("Invoice")["Invoice"].transform("count")
transactions_grouped = transactions_grouped[transactions_grouped["item_count"] > 1]

# create pairs of products in invoices group
basket_pairs = []
for invoice, group in transactions_grouped.groupby("Invoice"):
    items = group['Description'].unique()  # unique items in invoice
    pairs = list(combinations(sorted(items), 2))  # all pairs (sorted-> A,B == B,A)
    basket_pairs.extend(pairs)

# top 10 of repeating products
pair_counts = pd.Series(basket_pairs).value_counts().head(10).reset_index()
pair_counts[['product_a', 'product_b']] = pd.DataFrame(pair_counts['index'].tolist(), index=pair_counts.index)
pair_counts = pair_counts[["product_a","product_b","count"]].rename(columns={"count":"times_purchased_together"})

### Answer

In [ ]:
display(pair_counts)
display(md(f"Table below shows top 10 products pairs. Products A and B are often purchased together, so they can be promoted as a bundle. Customers who buy A often add B, so it is worth displaying them next to each other."))

## 4.4. What is the average order value? How does it change over time? Are there regions/countries with higher AOV?

In [ ]:
# revenue sum per invoice
col_dict = {
    "InvoiceDate":"first",
    "InvoiceDate_YYYYMM":"first",
    "Customer ID":"first",
    "Country":"first",
    "Category":"first",
    "Revenue":"sum"
    }

invoice_revenue = df.groupby("Invoice", as_index=False).agg(col_dict)

In [ ]:
# average order value
avg_order = round(invoice_revenue["Revenue"].mean(),2)

# average order over time
avg_order_by_month = invoice_revenue.groupby("InvoiceDate_YYYYMM", as_index=False)["Revenue"].mean()

# average order value per Country
avg_order_by_country = invoice_revenue.groupby("Country", as_index=False).agg(AverageRevenue=("Revenue", "mean"), InvoiceCount=("Invoice", "count")).sort_values(by="AverageRevenue", ascending=False)
avg_order_by_country["AverageRevenue"] = round(avg_order_by_country["AverageRevenue"],2)

### Answer

In [ ]:
sns.set_theme(
    style="white"
    )
f, (ax1, ax2) = plt.subplots(2,1, figsize=(10, 12))

(
    so.Plot(
        data=avg_order_by_month,
        x="InvoiceDate_YYYYMM",
        y="Revenue"
        )       
    .add(
        so.Bar(
            alpha=1, 
            color="#191f29"
            )
        )
    .scale(
        y=so.Continuous().tick(every=50)
        )   
    .label(
        title="Invoice average revenue over time | Mean: orange line",
        x="Date",
        y="Invoice revenue",
        legend=False
        )
    .on(ax1)
    .plot()
)

(
    so.Plot(
        data=avg_order_by_country,
        x="Country",
        y="AverageRevenue"
        )       
    .add(
        so.Bar(
            alpha=1, 
            color="#191f29"
            )
        )
    .scale(
        y=so.Continuous().tick(every=100)
        )   
    .label(
        title="Invoice average revenue per Country/Region | Mean: orange line",
        x="Country/Region",
        y="Invoice revenue",
        legend=False
        )
    .on(ax2)
    .plot()
)

ax1.axhline(y=(avg_order_by_month["Revenue"].mean() ), linewidth=2, color='orange', ls=':')
ax1.yaxis.set_tick_params(which="both", left=True, length=4, rotation=0)
ax1.yaxis.grid(True, linestyle="--", alpha=0.7)
ax1.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax1.yaxis.grid(True, linestyle="--", alpha=0.7)
ax1.set_axisbelow(True)

ax2.axhline(y=(avg_order_by_country["AverageRevenue"].mean() ), linewidth=2, color='orange', ls=':')
ax2.yaxis.set_tick_params(which="both", left=True, length=4, rotation=0)
ax2.yaxis.grid(True, linestyle="--", alpha=0.7)
ax2.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax2.yaxis.grid(True, linestyle="--", alpha=0.7)
ax2.set_axisbelow(True)

plt.tight_layout()
palette = sns.color_palette()
plt.show()

display(md("**Top 10 countries per Invoice revenue**"))
display(avg_order_by_country.head(10))

In [ ]:
display(md(f'''
  The "Invoice revenue over time" show the individual months in which the revenue above the reference level for the entire period presented was exceeded.\n
  Above the average, the period from the September to the December/January stands out clearly, ending abruptly.\n
  This can easily be linked to the Christmas and New Year period, when many retail sectors experience increased traffic.\n
  Like in general point of view - the increased number of invoices in May does not effect in Revenue per invoice. As it was seen: the number of invoices did not generate higher revenue, so it effects on low average revenue per invoice.
  '''))

display(md(f'''
  There are several countries with average invoice revenue higher than average. The attached table shows the top 10 countries.\n
  The highest invoice revenue is generated by **{avg_order_by_country.head(10).iloc[0,0]}** with a number of invoices amounting to **{avg_order_by_country.head(10).iloc[0,2]}**.
  '''))


## 4.5. Customer retention: how many customers return after their first purchase? What is the difference between “one-time” and “returning” customers in terms of value?

In [ ]:
# remove unkown customers
retention = df[df["Customer ID"].notna()]

# revenue sum per invoice
col_dict = {
    "InvoiceDate":"first",
    "InvoiceDate_YYYYMM":"first",
    "Customer ID":"first",
    "Country":"first",
    "Category":"first",
    "Revenue":"sum"
    }

retention = retention.groupby("Invoice", as_index=False).agg(col_dict).sort_values(by=["Customer ID", "InvoiceDate"], ascending=True)

# invoices per customer
order_count = retention["Customer ID"].value_counts().reset_index()

# customer types
all_customers = order_count["Customer ID"].count()
one_time_customers = order_count[order_count["count"] == 1]["Customer ID"].count()
one_time_ratio = round((one_time_customers / all_customers) * 100,1)
returning_customers = order_count[order_count["count"] > 1]["Customer ID"].count()
returning_ratio = round((returning_customers / all_customers) * 100,1)

# one-time and returning average revenue
one_time_customers_list = order_count.copy()
one_time_customers_list = one_time_customers_list[one_time_customers_list["count"] == 1]
one_time_customers_list["CustomerType"] = "one-time"
returning_customers_list = order_count.copy()
returning_customers_list = returning_customers_list[returning_customers_list["count"] > 1]
returning_customers_list["CustomerType"] = "returning"

customers_type = pd.concat((one_time_customers_list,returning_customers_list))
customers_type_revenue = pd.merge(retention, customers_type[["Customer ID", "CustomerType"]], on="Customer ID")
customers_type_revenue = customers_type_revenue.groupby("CustomerType", as_index=False)["Revenue"].mean().round(2)

In [ ]:
all_customers

In [ ]:
# Cohorts
cohorts = retention.copy()
cohorts["FirstOrder"] = cohorts.groupby("Customer ID")["InvoiceDate"].transform("min")
cohorts["CohortMonth"] = cohorts["FirstOrder"].dt.to_period("M")
cohorts["MonthDiff"] = (cohorts["InvoiceDate"].dt.year - cohorts["FirstOrder"].dt.year) * 12 + (cohorts["InvoiceDate"].dt.month - cohorts["FirstOrder"].dt.month)

cohorts = cohorts[["InvoiceDate", "FirstOrder", "MonthDiff", "CohortMonth", "Customer ID", "Invoice"]]
cohorts = cohorts.groupby(["MonthDiff", "CohortMonth"], as_index=False)["Customer ID"].nunique().sort_values(by="MonthDiff", ascending=True)
cohorts_pivot = cohorts.pivot(index="CohortMonth", columns="MonthDiff", values="Customer ID")
first_month_size = cohorts_pivot.iloc[:, 0]

# % retention
retention_matrix = cohorts_pivot.divide(first_month_size, axis=0)
retention_matrix = retention_matrix * 100

In [ ]:
plt.figure(figsize=(20, 8))
ax = sns.heatmap(retention_matrix, 
            annot=True,
            fmt=".0f",
            cmap='YlGnBu',
            cbar_kws={'label': 'Retention Rate [%]'})
ax.set(xlabel="Return month", ylabel="First order month")
plt.title('Customer retention (cohorts)')
plt.show()

### Answer

In [ ]:
display(md(f"Of all **{all_customers}** known customers (excluding unknown ‘Customer ID’), **{one_time_customers}** made one-time purchases, which accounts for **{one_time_ratio}%**."))
display(md(f"**{returning_customers}** made subsequent purchases, which represents **{returning_ratio}%**. It can be said that customer retention is high."))
display(md(f"One-time customers place orders worth an average of **{customers_type_revenue.loc[customers_type_revenue["CustomerType"] == "one-time", "Revenue"].item()}**"))                                                                             
display(md(f"Returning customers place orders worth an average of **{customers_type_revenue.loc[customers_type_revenue["CustomerType"] == "returning", "Revenue"].item()}**")) 
display(md(f"The heatmap from the cohort analysis shows a certain pattern, indicating that customers return during the Christmas season. The date of the first order and the month of the next order most often fall within the Christmas season."))

# Summary

In [ ]:
display(md(f"An analysis of the Online Retail dataset showed that as many as **22.77%** of transactions lack a customer ID, which is a significant area for improvement in data collection quality. The key conclusion is the enormous concentration of revenue: just **10%** of customers (mainly from the UK) generate as much as **55%** of total sales. Seasonality is clearly evident, with a sharp increase in activity between September and January, which directly correlates with the holiday season. Despite an increase in the number of invoices in May, this does not translate into higher average revenue, suggesting periodic promotions of lower-margin products. The retention rate is high **(75.4%)**, and returning customers generate nearly **40%** higher average order value than one-time customers. Identified anomalies in prices and quantities (including negative values) are not errors, but reflect real business processes such as returns, fees, and write-offs. The use of detected correlations in shopping carts represents ready-made potential for sales optimization through cross-selling."))